In [1]:
import os, numpy as np
from google import genai
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
EMB, GEN = "gemini-embedding-001", "gemini-2.5-flash"
 
# 1. 문서 임베딩 
documents = [
    "청킹은 긴 문서를 검색에 적합한 작은 조각으로 나누는 작업이다.",
    "임베딩 모델은 보통 수백~수천 토큰까지만 한 번에 인코딩한다.",
    "오버랩은 이웃한 청크가 일부 겹치도록 잘라 문장 중간에서 맥락이 끊기는 문제를 줄인다.",
    "권장 청크 크기는 보통 300~800 토큰, 오버랩은 10~20% 수준이다.",
]
def embed(text):
    r = client.models.embed_content(model=EMB, contents=text)
    return np.array(r.embeddings[0].values)
doc_vecs = [embed(d) for d in documents]
 
# 2. 질문 임베딩 
question = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"
q_vec = embed(question)
 
# 3. 코사인 유사도로 Top-K 검색 
def cos(a, b): return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
scored = sorted(zip(documents, doc_vecs), key=lambda x: -cos(q_vec, x[1]))
top_k = [doc for doc, _ in scored[:2]]
 
# 4. 프롬프트 주입 + 답변 생성
prompt = f"다음 자료만 근거로 답하라.\n자료:\n" + "\n".join(top_k) + f"\n\n질문: {question}"
print(client.models.generate_content(model=GEN, contents=prompt).text)

제공된 자료에 따르면, 청킹에서 오버랩이 필요한 이유는 "문장 중간에서 맥락이 끊기는 문제를 줄이기 위함" 입니다.


In [2]:
import numpy as np
import ollama

EMB = "nomic-embed-text"
GEN = "llama3.2:3b"

# 1. 문서 임베딩
documents = [
    "청킹은 긴 문서를 검색에 적합한 작은 조각으로 나누는 작업이다.",
    "임베딩 모델은 보통 수백~수천 토큰까지만 한 번에 인코딩한다.",
    "오버랩은 이웃한 청크가 일부 겹치도록 잘라 문장 중간에서 맥락이 끊기는 문제를 줄인다.",
    "권장 청크 크기는 보통 300~800 토큰, 오버랩은 10~20% 수준이다.",
]

def embed(text):
    r = ollama.embed(model=EMB, input=text)
    return np.array(r['embeddings'][0])

doc_vecs = [embed(d) for d in documents]

# 2. 질문 임베딩
question = "청킹 전략 방식 4가지는 무엇인가요?"
q_vec = embed(question)

# 3. 코사인 유사도로 Top-K 검색
def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

scored = sorted(zip(documents, doc_vecs), key=lambda x: -cos(q_vec, x[1]))
top_k = [doc for doc, _ in scored[:2]]

# 4. 프롬프트 주입 + 답변 생성
prompt = f"다음 자료만 근거로 답하라.\n자료:\n" + "\n".join(top_k) + f"\n\n질문: {question}"
response = ollama.chat(
    model=GEN,
    messages=[{"role": "user", "content": prompt}]
)
print(response['message']['content'])

청キング 전략의 4가지 방법은 다음과 같습니다:

1. 구분 (Segmentation)
2. 오버랩 (Overlapping)
3. 마크다운 (Markdown)
4. XML (Extensible Markup Language)
